# Misc

> A few small user-friendly components.

In [ ]:
#| default_exp misc

In [ ]:
#| export

import sympy as sp

from SFCmodeler.term import Variable, Term

In [ ]:
#| export

def if_true(conditional):
  return sp.Piecewise((1,conditional),(0,True))

`if_true` is for quickly creating conditional expressions. The input is a `sympy` comparison, such as an inequality, and the output is a $1$ if that comparison is true or 0 if false. It works by creating a piecewise function. E.g.

In [ ]:
BLR = Variable("BLR", "Bank liquidity ratio")
z1 = Variable("z_1","Bank liquidity indicator")

z1 << if_true(BLR > 0.12)

Eq(z_1, Piecewise((1, BLR > 0.12), (0, True)))

In [ ]:
#| export

def lag(x:Term | sp.Expr | float, n:int=1) -> sp.Expr:
    "Lag every Term in `x` by n periods (x can be a Term, expression, or constant)."
    x = sp.sympify(x)
    return x.xreplace({t: t(-n) for t in x.free_symbols if isinstance(t, Term)})

In [ ]:
V = Variable("V","Wealth")
Y = Variable("Y","Income")
C = Variable("C","Consumption")

lag(V - (Y-C))

C_{-1} + V_{-1} - Y_{-1}

In [ ]:
#| hide

assert lag(V - (Y-C)) == V(-1) - Y(-1) + C(-1)

In [ ]:
#| export

class d:
    def __init__(self, x):
        self.term = x if isinstance(x, Variable) else None   # only a Variable can be given an equation
        self.expr = x - lag(x) if lag(x) else sp.S.Zero

    def __lshift__(self, rhs):
        if self.term is None:
            raise TypeError("`d(...) << rhs` needs a single term, e.g. d(Y) << ..., not an expression")
        return self.term << self.term(-1) + rhs

    # lets sympy accept it anywhere a sympy expression is expected (Y - d(V), sp.Eq, ...)
    def _sympy_(self):
        return self.expr

    def _repr_latex_(self):
        return self.expr._repr_latex_()

    def _latex(self, printer):
        return printer.doprint(self.expr)

    # right-hand side arithmetic: 2 * d(Y), d(Y) / Y(-1), ...
    def __add__(self, o):      return self.expr + o
    def __radd__(self, o):     return o + self.expr
    def __sub__(self, o):      return self.expr - o
    def __rsub__(self, o):     return o - self.expr
    def __mul__(self, o):      return self.expr * o
    def __rmul__(self, o):     return o * self.expr
    def __truediv__(self, o):  return self.expr / o
    def __rtruediv__(self, o): return o / self.expr
    def __neg__(self):         return -self.expr
    def __eq__(self,o):        return self.expr == sp.sympify(o)
    def __repr__(self):        return repr(self.expr)

`d` allows for a first difference. With a single `Variable`, You can use it on either the left or right side of an equation:

In [ ]:
d(V) << Y - C

Eq(V, -C + V_{-1} + Y)

In [ ]:
D = Variable("D","Stock of debt")
NL = Variable("NL","New lending this period")

NL << d(D)

Eq(NL, D - D_{-1})

It works on more complicated expression too, though must be on the right side:

In [ ]:
d(Y - C)

-C + C_{-1} + Y - Y_{-1}

In [ ]:
d(2)

0

In [ ]:
#| hide

assert d(D) == D - D(-1)
assert (d(V) << Y - C) == sp.Eq(V, V(-1) + Y - C) 
assert d(Y - C) == Y-Y(-1) - (C - C(-1))
assert d(2) == 0 
assert d(2) == sp.S.Zero

try:
    d(Y - C) << 5
    assert False # can't do expression on LHS
except:
    pass

try:
    d(5) << 10
    assert False # can't do number on LHS
except:
    pass

In [ ]:
#| export

def solve_for(expression, term):
    "Solves a zero expression for a specified variable."
    solutions = sp.solve(expression, term)
    if len(solutions) == 1: return sp.Eq(term, solutions[0])
    else: raise RuntimeError(f"{len(solutions)} solutions found for {term}")

In [ ]:
solve_for(d(V) - (Y - C), V)

Eq(V, -C + V_{-1} + Y)

This is useful for the `Representations` classes that returnn `zero_expressions`.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()